#  Feature Engineering

##  Objective
The goal of this notebook is to transform raw cleaned data into meaningful features that can be used for building a robust Dynamic Pricing Model.

## 🔍 Key Tasks:
- Feature creation
- Feature transformation
- Encoding categorical variables
- Preparing final dataset for modeling

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df = pd.read_csv(r"C:\Users\hp\Desktop\PROJECTS\Dynamic_Pricing_Engine\data\processed\ecommerce_cleaned.csv")

df.head()

### converting event_time to datetime format 

In [ ]:
df['event_time'] = pd.to_datetime(df['event_time'], errors='coerce')

In [ ]:
df = df.drop(columns=['event_time'])

##  Handling Time Feature

The `event_time` column was initially present in object format and converted to datetime for consistency.

However, due to limited temporal coverage in the dataset (only a few hours of a single day), time-based features were deemed unreliable.

Therefore, `event_time` is excluded from feature engineering and modeling to ensure data quality and model robustness.

##  Selecting Relevant Features

In [ ]:
df = df[['event_type', 'product_id', 'price', 'user_id']]

df.head()

##  Price Transformation

In [ ]:
# Log transformation (MAIN FEATURE)
df['log_price'] = np.log1p(df['price'])

## Target for classification

In [ ]:
df['is_purchase'] = df['event_type'].apply(lambda x: 1 if x == 'purchase' else 0)

In [ ]:
product_stats = df.groupby('product_id').agg({   # all rows by each product
    'price': ['mean', 'std'], #average price & how much price varies(std)
    'is_purchase': 'sum', #total number of purchases per product
    'event_type': 'count'
}).reset_index()

product_stats.columns = [
    'product_id',
    'avg_price',
    'price_std',
    'total_purchases',
    'total_events'
]

In [ ]:
product_stats

In [ ]:
df = df.merge(product_stats, on='product_id', how='left')

In [ ]:
user_stats = df.groupby('user_id').agg({
    'event_type': 'count',
    'is_purchase': 'sum'
}).reset_index()

user_stats.columns = [
    'user_id',
    'user_activity',
    'user_purchases'
]

In [ ]:
df = df.merge(user_stats, on='user_id', how='left')

In [ ]:
df['conversion_rate'] = df['total_purchases'] / df['total_events']

In [ ]:
df['price_deviation'] = df['price'] - df['avg_price']

In [ ]:
df = pd.get_dummies(df, columns=['event_type'], drop_first=True)

In [ ]:
#dropping unnecessary columns
df = df.drop(columns=['product_id', 'user_id'])

In [ ]:
dummy_cols = [col for col in df.columns if 'event_type_' in col]
df[dummy_cols] = df[dummy_cols].astype(int)

In [ ]:
df = df.drop(columns=['event_type_purchase'])

In [ ]:
df.columns

In [ ]:
df['price_std'] = df['price_std'].fillna(0)

In [ ]:
#final check 
print('---------------------------------------------------------')
print('sample rows:',df.head())
print('---------------------------------------------------------')
print('data summary',df.info())
print('---------------------------------------------------------')
print('shape of data :',df.shape)

In [ ]:
df.to_csv(r"C:\Users\hp\Desktop\PROJECTS\Dynamic_Pricing_Engine\data\final\final_featured_data.csv", index=False)